In [1]:
import os
import certifi
import requests

from pydantic_settings import BaseSettings
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from langchainhub import Client

C:\Users\Admin\AppData\Local\Temp\ipykernel_16936\1743884499.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
d:\test-aie\llm-app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.agents import create_agent

class Settings(BaseSettings):
    GEMINI_API_KEY: str | None = None
    TAVILY_API_KEY: str | None = None
    model_config = {"env_file": ".env", "extra": "ignore"}
    
    
settings = Settings()

In [31]:
search_tool = TavilySearchResults(max_results=3, api_key=settings.TAVILY_API_KEY)

In [6]:
search_tool.invoke("Thủ đô của Việt Nam là tỉnh nào?")

[{'title': 'Thủ đô Việt Nam – Wikipedia tiếng Việt',
  'url': 'https://vi.wikipedia.org/wiki/Th%E1%BB%A7_%C4%91%C3%B4_Vi%E1%BB%87t_Nam',
  'content': 'Hiện nay Việt Nam có 5 tỉnh, thành được gọi là các "vùng kinh đô" gồm: Hà Nội, Phú Thọ, Ninh Bình, Thanh Hóa và Thừa Thiên – Huế. 5 vùng kinh đô này được ngành văn hóa cho phép tổ chức và tham gia nhiều sự kiện lớn như: cuộc thi người đẹp các vùng kinh đô, hiệp hội văn học nghệ thuật các vùng kinh đô, triển lãm ảnh ngũ đại cố đô của Việt Nam, Hành trình di sản thế giới... Năm du lịch Quốc gia 2015 diễn ra ở Thanh Hóa và các tỉnh, thành phố có Kinh đô cổ và di sản văn hoá thế giới có chuyên đề "Hành trình về Kinh đô cổ Việt Nam".\n\n## Xem thêm\n\n## Tham khảo\n\nWikimedia Foundation\nPowered by MediaWiki [...] ## Cố đô của Việt Nam\n\nCố đô là cách gọi tôn vinh những nơi từng là thủ đô chính thống trong lịch sử Việt Nam. Hiện ở Việt Nam có các nơi sau được gọi là cố đô gồm: đất tổ Phong Châu "Phong Châu (kinh đô)"), cố đô Hoa Lư "Hoa Lư 

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite", 
    api_key=settings.GEMINI_API_KEY,
    temperature=0.0)

In [9]:
response =llm.invoke("Hôm nay là ngày mấy?")
response

AIMessage(content=[{'type': 'text', 'text': 'Hôm nay là **thứ Ba, ngày 22 tháng 10 năm 2024**.', 'extras': {'signature': 'EjQKMgERTTIPNbln9x4/ookjxHsFc/dFvojP27GrYG8oZ8QJsH2iGWW+QBrsvJKyDOmGCZqw'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fc6a3-aec4-7cd0-96bd-8c1df68e8a4c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 24, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}})

In [84]:
#=====================
# TẠO PROMPT TEMPLATE
#=====================

from langchain_core.prompts import PromptTemplate



# Định nghĩa template chuỗi (nội dung chi tiết có tại hwchase17/react)
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

# Khởi tạo PromptTemplate từ chuỗi
prompt = PromptTemplate.from_template(react_template)

In [77]:
import requests
@tool
def get_weather_data(city: str):
    """lấy nhiệt độ hiện tại của một thành phố"""
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": city,
            "count": 1,
            "language": "vi",
            "format": "json"
        }
    ).json()

    if not geo.get("results"):
        raise ValueError(f"City '{city}' not found.")

    location = geo["results"][0]

    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": location["latitude"],
            "longitude": location["longitude"],
            "current_weather": "true"
        }
    ).json()
    print(weather)
    temperature = weather.get("current_weather", {}).get("temperature")
    return {"city": city, "temperature": temperature}

# get_weather_data("Đà Nẵng")

In [93]:
#=====================
# TẠO TOOLS
#=====================

tools = [search_tool, get_weather_data]

In [96]:
#=====================
# TAO AGENT
#=====================
from langchain_classic.agents import create_react_agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

In [ ]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Hãy search cho mình, thành phố trực thuộc trung ương miền trung Việt Nam là thành phố nào và nhiệt độ hiện tại của thành phố đó là bao nhiêu?"}
    ]
})

In [81]:
for msg in response["messages"]:
    if getattr(msg, "tool_calls", None):
        print("TOOL CALL:", msg.tool_calls)
print("Agent Response:", response["messages"][-1].content)

TOOL CALL: [{'name': 'tavily_search_results_json', 'args': {'query': 'thành phố trực thuộc trung ương miền trung Việt Nam'}, 'id': 'k5BqXAu7', 'type': 'tool_call'}]
TOOL CALL: [{'name': 'get_weather_data', 'args': {'city': 'Đà Nẵng'}, 'id': 'K0dSp0pQ', 'type': 'tool_call'}]
Agent Response: [{'type': 'text', 'text': 'Thành phố trực thuộc trung ương nằm ở miền Trung Việt Nam hiện nay là **Đà Nẵng**.\n\nNhiệt độ hiện tại của thành phố Đà Nẵng là khoảng **29.6°C**.', 'extras': {'signature': 'EjQKMgERTTIP2JzP/9t9g7skdz3g/hxChIRqF1XB1ljHafPJsIcNU/Z4TgoXCe90oe7NNodu'}}]


In [107]:
from langchain_classic.agents import AgentExecutor

agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    return_intermediate_steps=True,
    handle_parsing_errors=True)

result = agent_executor.invoke({
    "input": "Thành phố trực thuộc trung ương miền trung Việt Nam là thành phố nào và nhiệt độ hiện tại của thành phố đó là bao nhiêu?"
})



> Entering new AgentExecutor chain...
Thought: Tôi cần xác định thành phố trực thuộc trung ương ở miền Trung Việt Nam, sau đó lấy nhiệt độ hiện tại của thành phố đó.

Action: tavily_search_results_json
Action Input: "thành phố trực thuộc trung ương miền trung Việt Nam"
[{'title': 'Thành phố trực thuộc trung ương (Việt Nam) – Wikipedia tiếng Việt', 'url': 'https://vi.wikipedia.org/wiki/Th%C3%A0nh_ph%E1%BB%91_tr%E1%BB%B1c_thu%E1%BB%99c_trung_%C6%B0%C6%A1ng_(Vi%E1%BB%87t_Nam)', 'content': 'Trung tm Hà Nội")  Trung tm Hải Phòng")  Trung tm Huế "Huế (thành phố thuộc tỉnh)")  Trung tm Thành phố Hồ Chí Minh")  Tuyên Quang "Tuyên Quang (thành phố)")  Tuy Hòa "Tuy Hòa (thành phố)")  Từ Sơn "Từ Sơn (thành phố)")  Uông Bí "Uông Bí (thành phố)")  Vị Thanh "Vị Thanh (thành phố)")  Việt Trì "Việt Trì (thành phố)")  Vinh  Vĩnh Long "Vĩnh Long (thành phố)")  Vĩnh Yên "Vĩnh Yên (thành phố)")  Vũng Tàu "Vũng Tàu (thành phố)")  Yên Bái "Yên Bái (thành phố)") | [...] "Sa Đéc (thành phố)")  Sầm Sơn "Sầm 

In [ ]:
result["output"]
result["intermediate_steps"]

[(AgentAction(tool='tavily_search_results_json', tool_input='thành phố trực thuộc trung ương miền trung Việt Nam"\n', log='Thought: Tôi cần xác định thành phố trực thuộc trung ương ở miền Trung Việt Nam, sau đó lấy nhiệt độ hiện tại của thành phố đó.\n\nAction: tavily_search_results_json\nAction Input: "thành phố trực thuộc trung ương miền trung Việt Nam"\n'),
  [{'title': 'Thành phố trực thuộc trung ương (Việt Nam) – Wikipedia tiếng Việt',
    'url': 'https://vi.wikipedia.org/wiki/Th%C3%A0nh_ph%E1%BB%91_tr%E1%BB%B1c_thu%E1%BB%99c_trung_%C6%B0%C6%A1ng_(Vi%E1%BB%87t_Nam)',
    'content': 'Trung tm Hà Nội")  Trung tm Hải Phòng")  Trung tm Huế "Huế (thành phố thuộc tỉnh)")  Trung tm Thành phố Hồ Chí Minh")  Tuyên Quang "Tuyên Quang (thành phố)")  Tuy Hòa "Tuy Hòa (thành phố)")  Từ Sơn "Từ Sơn (thành phố)")  Uông Bí "Uông Bí (thành phố)")  Vị Thanh "Vị Thanh (thành phố)")  Việt Trì "Việt Trì (thành phố)")  Vinh  Vĩnh Long "Vĩnh Long (thành phố)")  Vĩnh Yên "Vĩnh Yên (thành phố)")  Vũng Tàu

In [101]:
from langgraph.prebuilt import create_react_agent

langraph_agent = create_react_agent(
    model=llm,
    prompt=react_template,
    tools=tools,
)

langraph_result = langraph_agent.invoke({
    "messages": [
        {"role": "user", "content": "Hãy search cho mình, hôm nay là ngày mấy, thông tin về chiến sự tại Iran và giá vàng hôm nay"}
    ]
})


C:\Users\Admin\AppData\Local\Temp\ipykernel_16936\3939786095.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  langraph_agent = create_react_agent(


In [105]:
for msg in langraph_result["messages"]:
    if getattr(msg, "tool_calls", None):
        print("TOOL CALL:", msg.tool_calls)
        
langraph_result["messages"][-1].content

TOOL CALL: [{'name': 'tavily_search_results_json', 'args': {'query': 'hôm nay là ngày mấy, tình hình chiến sự Iran mới nhất, giá vàng hôm nay tại Việt Nam'}, 'id': 'oN8J9BLf', 'type': 'tool_call'}]


[{'type': 'text',
  'text': 'Hôm nay là ngày **thứ Tư, ngày 5 tháng 8 năm 2026**.\n\nDưới đây là thông tin tổng hợp về các vấn đề bạn quan tâm:\n\n### 1. Tình hình chiến sự tại Iran\nTheo các thông tin cập nhật gần đây, tình hình căng thẳng giữa Mỹ và Iran đã có những chuyển biến tích cực. Các bên đã thông báo thống nhất được các điều khoản để **chấm dứt chiến sự** và mở cửa lại eo biển Hormuz. Thông tin về việc hạ nhiệt căng thẳng này đã có tác động trực tiếp đến thị trường tài chính toàn cầu.\n\n### 2. Giá vàng hôm nay (5/8/2026)\nThông tin về giá vàng đang có những biến động đáng chú ý sau khi có tin tức về việc chấm dứt chiến sự:\n*   **Giá vàng thế giới:** Có xu hướng tăng nhẹ. Ghi nhận vào sáng nay, giá vàng thế giới giao dịch quanh mốc **4.340 USD/ounce**.\n*   **Giá vàng trong nước:** Tại một số thương hiệu lớn (như Bảo Tín Minh Châu), giá vàng SJC và vàng Rồng Thăng Long đang giao dịch ở mức khoảng **149,80 triệu đồng/lượng (mua vào) – 151,80 triệu đồng/lượng (bán ra)**.\n\n*L

In [50]:
#=====================
# TAO AGENT - API MOI (langchain.agents.create_agent)
# create_agent la API moi thay the create_react_agent (deprecated)
#=====================
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. Use the provided tools to fetch "
        "up-to-date information when it is needed. Always respond in Vietnamese."
    ),
)

In [51]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Hãy search cho mình, hôm nay là ngày mấy, thông tin về chiến sự tại Iran và giá vàng hôm nay"}
    ]
})

In [52]:
for msg in response["messages"]:
    if getattr(msg, "tool_calls", None):
        print("TOOL CALL:", msg.tool_calls)
print("Agent Response:", response["messages"][-1].content)

TOOL CALL: [{'name': 'tavily_search_results_json', 'args': {'query': 'hôm nay là ngày mấy dương lịch'}, 'id': 'I5JmjpKN', 'type': 'tool_call'}, {'name': 'tavily_search_results_json', 'args': {'query': 'tình hình chiến sự Iran mới nhất hôm nay'}, 'id': 'cgpZP726', 'type': 'tool_call'}, {'name': 'tavily_search_results_json', 'args': {'query': 'giá vàng hôm nay tại Việt Nam'}, 'id': 'ebfZan4O', 'type': 'tool_call'}]
Agent Response: [{'type': 'text', 'text': 'Chào bạn, dưới đây là thông tin cập nhật cho bạn:\n\n### 1. Hôm nay là ngày mấy?\nHôm nay là **thứ Tư, ngày 05 tháng 08 năm 2026** (dương lịch).\n\n### 2. Thông tin về tình hình tại Iran\nCác thông tin về tình hình tại Iran hiện nay đang có nhiều diễn biến phức tạp liên quan đến căng thẳng địa chính trị và quân sự:\n*   **Căng thẳng quân sự:** Các báo cáo ghi nhận tình trạng căng thẳng kéo dài giữa Iran và Mỹ, bao gồm các cuộc không kích, các đợt điều động quân sự và cảnh báo từ cả hai phía.\n*   **An ninh hàng hải:** Có các thông tin